# Stage 4: TensorRT 11 on a Colab T4

Builds FP32, FP16 and INT8 engines of the MicroSAM Base encoder (1x3x1024x1024, batch 1) and compares
them with the FP32 ONNX Runtime reference. Run on a **T4 GPU** runtime (Runtime > Change runtime type).

**TensorRT 11 works differently from 10.x.** The `FP16` and `INT8` builder flags no longer exist and every
network is strongly typed: the precisions must already be inside the ONNX file.
- FP32: build the FP32 ONNX directly.
- FP16: convert the weights first (`onnxconverter-common`), then build.
- INT8: needs a QDQ model with **symmetric** quantization (zero point 0) and **float biases**. It is made on
  the laptop with `stage3_ort_int8.py quantize --variant trt_matmul`.

**Needs on Google Drive** (`MyDrive/edge_results/models/`): `sam_vitb_encoder_fp32_dynamo.onnx` + `.onnx.data`,
`sam_vitb_encoder_int8_trt_matmul.onnx`. **Upload by hand** to `/content/`: `trt_check_tiles.npz`.

**What the timings are:** GPU execution time of one inference (CUDA events), 10 warm-up and 30 timed runs,
median with p10/p90. Host-to-device copies are not included. Colab is a different machine from the
laptop: compare only ratios within the T4 results, never absolute times with the laptop.

Results are saved as JSON to `MyDrive/edge_results/results_colab/`. Copy them to `edge_deploy/results/`.

In [ ]:
!pip install -q tensorrt polygraphy onnx onnxruntime-gpu colored onnxconverter-common
import importlib.metadata as md, torch
for p in ["tensorrt", "onnx", "onnxruntime-gpu", "polygraphy", "onnxconverter-common"]:
    print(p, md.version(p))
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')

os.makedirs("/content/work/models", exist_ok=True)
D = "/content/drive/MyDrive/edge_results/models"
for f in ["sam_vitb_encoder_fp32_dynamo.onnx", "sam_vitb_encoder_fp32_dynamo.onnx.data",   # the .data file must sit next to the .onnx
          "sam_vitb_encoder_int8_trt_matmul.onnx"]:                                         # symmetric QDQ, made by stage3_ort_int8.py quantize --variant trt_matmul
    if not os.path.exists(f"/content/work/models/{f}"):
        subprocess.run(["cp", f"{D}/{f}", "/content/work/models/"], check=True)
print(subprocess.run("ls -la /content/work/models", shell=True, capture_output=True, text=True).stdout)
# Also upload trt_check_tiles.npz (made by stage4_export_check_tiles.py) to /content via the Files panel.

In [ ]:
import onnx, os
from onnxconverter_common import float16

src = "/content/work/models/sam_vitb_encoder_fp32_dynamo.onnx"
dst = "/content/work/models/sam_vitb_encoder_fp16.onnx"
m = onnx.load(src)                       # loads the .onnx.data weights too
m16 = float16.convert_float_to_float16(m, keep_io_types=True)
onnx.save(m16, dst)
print("fp16 model MB:", os.path.getsize(dst) / 1e6)
print("nodes:", len(m16.graph.node), "| ops:", sorted({n.op_type for n in m16.graph.node}))


In [ ]:
import tensorrt as trt, torch, json, time, os, subprocess, numpy as np
M = "/content/work/models"; E = "/content/work/engines"; os.makedirs(E, exist_ok=True)
R = "/content/drive/MyDrive/edge_results/results_colab"; os.makedirs(R, exist_ok=True)
logger = trt.Logger(trt.Logger.WARNING)

def clocks():
    return subprocess.run("nvidia-smi --query-gpu=clocks.sm,clocks.max.sm,temperature.gpu,power.draw --format=csv,noheader",
                          shell=True, capture_output=True, text=True).stdout.strip()

def build(onnx_path, plan_path):
    b = trt.Builder(logger)
    net = b.create_network()                      # strongly typed by default in TensorRT 11
    p = trt.OnnxParser(net, logger)
    if not p.parse_from_file(onnx_path):          # parse_from_file finds the .onnx.data next to it
        for i in range(p.num_errors): print(p.get_error(i))
        raise SystemExit("parse failed")
    cfg = b.create_builder_config()
    cfg.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 4 << 30)
    cfg.profiling_verbosity = trt.ProfilingVerbosity.DETAILED
    t = time.time()
    plan = b.build_serialized_network(net, cfg)
    if plan is None: raise SystemExit("build failed")
    open(plan_path, "wb").write(plan)
    return round(time.time() - t, 1)

TORCH = {trt.float32: torch.float32, trt.float16: torch.float16}

def bench(plan_path, runs=30, warm=10):
    eng = trt.Runtime(logger).deserialize_cuda_engine(open(plan_path, "rb").read())
    ctx = eng.create_execution_context()
    names = [eng.get_tensor_name(i) for i in range(eng.num_io_tensors)]
    g = torch.Generator().manual_seed(0); bufs = {}; out = None
    for n in names:
        shp = tuple(eng.get_tensor_shape(n)); dt = TORCH[eng.get_tensor_dtype(n)]
        if eng.get_tensor_mode(n) == trt.TensorIOMode.INPUT:
            bufs[n] = torch.randn(shp, generator=g).to(dt).cuda()
        else:
            bufs[n] = torch.empty(shp, dtype=dt, device="cuda"); out = n
        ctx.set_tensor_address(n, bufs[n].data_ptr())
    s = torch.cuda.Stream(); ms = []
    for i in range(warm + runs):
        a, z = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        a.record(s); ctx.execute_async_v3(s.cuda_stream); z.record(s); z.synchronize()
        if i >= warm: ms.append(a.elapsed_time(z))
    ms = np.array(ms)
    info = json.loads(eng.create_engine_inspector().get_engine_information(trt.LayerInformationFormat.JSON))
    layers = info.get("Layers", [])
    half = sum("Half" in json.dumps(l) for l in layers)
    return dict(median_ms=round(float(np.median(ms)), 2), p10_ms=round(float(np.percentile(ms, 10)), 2),
                p90_ms=round(float(np.percentile(ms, 90)), 2), runs=runs, warmup=warm,
                plan_MB=round(os.path.getsize(plan_path) / 1e6, 1), n_layers=len(layers), layers_with_half=half,
                out_dtype=str(bufs[out].dtype)), bufs[out].float().cpu().flatten()

def count(plan_path, key):
    """Number of engine layers whose description mentions `key` (e.g. "Half", "Int8")."""
    eng = trt.Runtime(logger).deserialize_cuda_engine(open(plan_path, "rb").read())
    info = json.loads(eng.create_engine_inspector().get_engine_information(trt.LayerInformationFormat.JSON))
    return sum(key in json.dumps(l) for l in info.get("Layers", []))


In [ ]:
results, outs = {}, {}
for name, f in [("fp32", "sam_vitb_encoder_fp32_dynamo.onnx"), ("fp16", "sam_vitb_encoder_fp16.onnx")]:
    path = f"{M}/{f}"
    assert os.path.exists(path), f"missing {path}: rerun the FP16 conversion cell"
    print(f"--- {name}: ONNX {os.path.getsize(path)/1e6:.1f} MB | clocks before: {clocks()}")
    t = build(path, f"{E}/{name}.plan")
    r, o = bench(f"{E}/{name}.plan"); r["build_s"] = t; r["clocks_after"] = clocks()
    results[name], outs[name] = r, o
    print(name, r)

cos = torch.nn.functional.cosine_similarity(outs["fp32"], outs["fp16"], dim=0).item()
results["sanity_cosine_fp16_vs_fp32_random_input"] = cos
results["versions"] = dict(tensorrt=trt.__version__, torch=torch.__version__, gpu=torch.cuda.get_device_name(0))
json.dump(results, open(f"{R}/stage4_trt_fp32_fp16.json", "w"), indent=2)
print("cosine FP16 vs FP32 (random input, sanity only):", cos)


In [ ]:
path = f"{M}/sam_vitb_encoder_int8_trt_matmul.onnx"
print("ONNX MB:", round(os.path.getsize(path) / 1e6, 1), "| clocks before:", clocks())

t = build(path, f"{E}/int8_trt.plan")
r, o = bench(f"{E}/int8_trt.plan")
r.update(build_s=t, clocks_after=clocks(), layers_with_int8=count(f"{E}/int8_trt.plan", "Int8"))
print("int8_trt", r)
cos = torch.nn.functional.cosine_similarity(outs["fp32"], o, dim=0).item()
print("cosine INT8 vs FP32 (random input, sanity only):", cos)
results["int8_trt"], results["sanity_cosine_int8_trt_vs_fp32_random_input"] = r, cos
json.dump(results, open(f"{R}/stage4_trt_all.json", "w"), indent=2)

In [ ]:
src = "/content/trt_check_tiles.npz"
d = np.load(src); imgs, ref = d["images"], d["reference"]
print("tiles", imgs.shape, "reference", ref.shape)

def encode(plan_path, x_np):
    eng = trt.Runtime(logger).deserialize_cuda_engine(open(plan_path, "rb").read())
    ctx = eng.create_execution_context(); bufs = {}; out = None
    for n in [eng.get_tensor_name(i) for i in range(eng.num_io_tensors)]:
        shp = tuple(eng.get_tensor_shape(n)); dt = TORCH[eng.get_tensor_dtype(n)]
        if eng.get_tensor_mode(n) == trt.TensorIOMode.INPUT:
            bufs[n] = torch.zeros(shp, dtype=dt, device="cuda"); inp = n
        else:
            bufs[n] = torch.empty(shp, dtype=dt, device="cuda"); out = n
        ctx.set_tensor_address(n, bufs[n].data_ptr())
    s = torch.cuda.Stream(); res = []
    for x in x_np:
        bufs[inp].copy_(torch.from_numpy(x[None]).to(bufs[inp].dtype))
        ctx.execute_async_v3(s.cuda_stream); s.synchronize()
        res.append(bufs[out].float().cpu().numpy()[0])
    return np.stack(res)

def compare(a, b):
    a, b = a.astype(np.float64).ravel(), b.astype(np.float64).ravel()
    return dict(cosine=float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b))),
                max_abs_diff=float(np.abs(a - b).max()), mean_abs_diff=float(np.abs(a - b).mean()))

check = {}
for name in ["fp32", "fp16", "int8_trt"]:
    out = encode(f"{E}/{name}.plan", imgs)
    tiles = [compare(ref[i], out[i]) for i in range(len(ref))]
    check[name] = dict(min_cosine=min(t["cosine"] for t in tiles),
                       max_abs_diff=max(t["max_abs_diff"] for t in tiles), tiles=tiles)
    print(f"{name:9s} min cosine {check[name]['min_cosine']:.5f} | worst max|diff| {check[name]['max_abs_diff']:.3f} | "
          f"per-tile cosines {[round(t['cosine'], 4) for t in tiles]}")
json.dump(check, open(f"{R}/stage4_trt_real_tile_check.json", "w"), indent=2)


## What did not work (kept for the write-up)

- **`trtexec` is not in the pip package**, so engines are built with the Python API.
- **`BuilderFlag.FP16` / `INT8` do not exist in TensorRT 11.** Use FP16-converted or QDQ ONNX models instead.
- **ModelOpt AutoCast crashed Colab (out of RAM).** Its reference run on the CPU keeps too much for this model;
  likely cause is the large attention intermediates (not confirmed). `onnxconverter-common` was used instead.
- **The Stage 3 QDQ model (`int8_static_matmul`) is rejected by TensorRT** for two reasons:
  Int32 bias `DequantizeLinear` nodes ("input has type Int32 but must have type ... Int8") and non-zero zero
  points on activations ("shiftIsAllZeros"). Folding the bias nodes fixed the first; the second needs a new
  symmetric quantization, hence `trt_matmul`.
- The full session log, including these failures, is in `stage4_session_log.ipynb`.